# TERF — ray tracing LOS/NLOS (stazione permanente)

Pipeline Colab per la stazione **TERF** (`TERF00CYP`, Cyprus).

**Input:** RINEX OBS `TERF00CYP_R_*_MO.rnx` (upload manuale o copia da repo).

**Output** in `/content/terf_work/results/`:
- mesh OSM (`terf_osm_triangles.npy`, `terf_osm.glb`)
- `reference_YYYYDOY.csv` (ECEF fisso)
- `terf_los_labels_YYYYDOY_gc.csv` (G+C)
- **`terf_los_viz_YYYYDOY.html`** — viewer Cesium 3D: raggi per satellite **osservato nel RINEX** (come Odaiba), verde=LOS / rosso=NLOS, click → grafico C/N₀
- `terf_pipeline_summary.json`

**Runtime:** GPU (T4) + build CUDA `_bvh`.

**Cesium ion:** token gratuito su https://cesium.com/ion/signup (terreno + contesto globo).

In [ ]:
# 1) Verifica GPU
!nvidia-smi

In [ ]:
# 2) Clone repo (branch feature/cuda-preprocess-area-map — main non contiene ancora TERF)
import os
REPO = "/content/gnss_gpu"
BRANCH = "feature/cuda-preprocess-area-map"
if os.path.isdir(REPO):
    %cd {REPO}
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/andrea444460/gnss_gpu_los_nlos.git {REPO}
    %cd {REPO}
!test -f experiments/run_terf_permanent_station_colab.py && echo "OK: script trovato"

In [ ]:
# 3) Dipendenze + build CUDA (T4 = sm_75)
!pip install -q numpy matplotlib folium branca pyproj rasterio requests scipy
!apt-get -qq install -y cmake
!mkdir -p build
%cd build
!cmake .. -DCMAKE_CUDA_ARCHITECTURES=75
!make -j$(nproc)
%cd ..

In [ ]:
# 4) Test import estensioni
import os, sys
os.environ["PYTHONPATH"] = "python:build"
sys.path[:0] = ["python", "build"]
import gnss_gpu._bvh
print("_bvh OK")

In [ ]:
# 5a) Copia RINEX dal repo (se presenti in experiments/data/TERF) + setup
import os
os.environ["PYTHONPATH"] = "python:build"
!mkdir -p /content/terf_work/data
!python experiments/run_terf_permanent_station_colab.py --phase setup --work-dir /content/terf_work

In [ ]:
# 5b) Upload manuale TERF*.rnx (se non nel repo)
from google.colab import files
from pathlib import Path

dest = Path("/content/terf_work/data")
dest.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    (dest / name).write_bytes(data)
    print("saved", dest / name)

In [ ]:
# 6) Labels geometriche (mesh + BRDC + CSV)
import os
os.environ["PYTHONPATH"] = "python:build"

# Test rapido: aggiungi --epoch-step 30
!python experiments/run_terf_permanent_station_colab.py \
  --work-dir /content/terf_work \
  --phase all \
  --systems G,C \
  --epoch-step 1

In [ ]:
# 7) Token Cesium ion
import os
os.environ["CESIUM_ION_TOKEN"] = "PASTE_YOUR_CESIUM_ION_TOKEN_HERE"

In [ ]:
# 8) Visualizzazione 3D per-epoch (solo satelliti nel RINEX OBS, come Odaiba)
import os
os.environ["PYTHONPATH"] = "python:build"

!python experiments/run_terf_permanent_station_colab.py \
  --work-dir /content/terf_work \
  --phase viz \
  --viz-day 2026192 \
  --n-epochs-viz 24 \
  --epoch-min-interval-s 600 \
  --obs-match-tol-s 20

In [ ]:
# 9) Apri viewer HTML (HTTP per GLB sidecar)
from pathlib import Path
from IPython.display import IFrame
import subprocess, time

html = Path("/content/terf_work/results/terf_los_viz_2026192.html")
if not html.exists():
    raise FileNotFoundError(html)

subprocess.Popen(
    "python -m http.server 8765 --directory /content/terf_work/results",
    shell=True,
)
time.sleep(1)
IFrame(src="http://localhost:8765/terf_los_viz_2026192.html", width="100%", height=700)

In [ ]:
# 10) Riepilogo
from IPython.display import JSON, display
import json
from pathlib import Path

summary_path = Path("/content/terf_work/results/terf_pipeline_summary.json")
if summary_path.exists():
    display(JSON(json.loads(summary_path.read_text())))

for p in sorted(Path("/content/terf_work/results").glob("terf_los_*")):
    print(p.name, p.stat().st_size, "bytes")

In [ ]:
# 11) Scarica zip
!zip -r /content/terf_results.zip /content/terf_work/results
from google.colab import files
files.download("/content/terf_results.zip")

## Fasi singole (debug)

```bash
python experiments/run_terf_permanent_station_colab.py --phase labels --work-dir /content/terf_work --epoch-step 30

python experiments/run_terf_permanent_station_colab.py --phase viz --work-dir /content/terf_work \
  --viz-day 2026192 --n-epochs-viz 12 --epoch-min-interval-s 1200

python experiments/run_terf_permanent_station_colab.py --phase all --with-viz --viz-day 2026192
```

Nel viewer: **Play** anima le epoche; click su un raggio → grafico C/N₀ del satellite.